# DocsMind lab: label the posts that actually answer each question

Pipeline position: **Search → Human relevance judgement → Retrieval evaluation**.

The first embedding benchmark measured whether a model could find a post describing the problem. This lab upgrades the labels: you inspect the thread and select posts containing a useful answer. No model or API decides relevance for you.


## Labelling rule

Select a post only when its text helps answer the evaluation question without guessing. A useful diagnostic suggestion counts; a reply saying only *same problem*, *thanks*, or *fixed now* does not. Multiple posts may be relevant.

Use `None` when the thread contains no verified answer. Leave `[]` while a question is still pending. The exporter refuses to create a supposedly reviewed dataset while any `[]` remains.

In an interview, this is the depth signal: retrieval quality is measured against human relevance judgements, not topic titles or synthetic labels silently produced by another model.


In [ ]:
from collections import defaultdict
from html import escape
from pathlib import Path
import json
import sys

import pandas as pd
from IPython.display import HTML, Markdown, display

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / 'docsmind').exists():
    ROOT = ROOT.parent
if not (ROOT / 'docsmind').exists():
    raise RuntimeError('Start Jupyter from the DocsMind repository.')
sys.path.insert(0, str(ROOT))

from docsmind.eval.answer_review import build_reviewed_answer_eval, rank_answer_candidates
from docsmind.eval.embedding_retrieval import load_embedding_eval
from docsmind.ingestion.briskoda_chunks import load_briskoda_posts

SNAPSHOT_PATH = Path.home() / 'projects/docsmind-data/briskoda/experiments/chunking-v1/snapshot/superb_mk3.briskoda.jsonl'
STARTER_EVAL_PATH = ROOT / 'data/eval/briskoda_embedding_queries.v1.json'
LAB_DIR = Path.home() / 'projects/docsmind-data/briskoda/experiments/answer-labels-v1'
REVIEWED_EVAL_PATH = LAB_DIR / 'briskoda_answer_queries.v1.json'
LAB_DIR.mkdir(parents=True, exist_ok=True)


## 1. Load the frozen corpus and review queue

The snapshot and query version are fixed. That ensures a label still refers to the same post when the crawler later resumes.


In [ ]:
posts = load_briskoda_posts(SNAPSHOT_PATH)
starter = load_embedding_eval(STARTER_EVAL_PATH)
queries = starter['queries']
queries_by_id = {item['id']: item for item in queries}
posts_by_id = {str(post['post_id']): post for post in posts}
posts_by_topic = defaultdict(list)
for post in posts:
    posts_by_topic[str(post['topic_id'])].append(post)
for topic_posts in posts_by_topic.values():
    topic_posts.sort(key=lambda post: (int(post.get('post_number') or 0), str(post.get('posted_at', ''))))

queue = pd.DataFrame([
    {
        'id': item['id'],
        'question': item['question'],
        'topic_posts': len(posts_by_topic[str(item['topic_id'])]),
    }
    for item in queries
])
print(f'Corpus posts: {len(posts):,}')
print(f'Questions to review: {len(queries)}')
display(queue)


## 2. Inspect one question and its reply candidates

Change `QUERY_ID`, then run the cell. BM25 surfaces vocabulary matches and the earliest replies are always included. This ranking only saves reading time; it does not create the labels. Set `SHOW_ALL_REPLIES=True` when the suggested candidates do not contain a convincing answer.


In [ ]:
def display_post(post, heading, extra=''):
    url = escape(str(post.get('post_url', '')), quote=True)
    text = escape(str(post.get('text', '')))
    metadata = (
        f"post_id={escape(str(post['post_id']))} | "
        f"post #{int(post.get('post_number') or 0)} | "
        f"author={escape(str(post.get('author', '')))} {escape(extra)}"
    )
    display(HTML(
        f"<h4>{escape(heading)}</h4><p><b>{metadata}</b> | "
        f"<a href='{url}' target='_blank'>open original post</a></p>"
        f"<pre style='white-space:pre-wrap;max-height:420px;overflow:auto'>"
        f"{text}</pre>"
    ))


def show_review(query_id, show_all_replies=False):
    item = queries_by_id[query_id]
    topic_posts = posts_by_topic[str(item['topic_id'])]
    display(Markdown(f"## {item['id']}\n\n**Question:** {item['question']}\n\n**Thread posts:** {len(topic_posts)}"))
    for source_id in item['relevant_post_ids']:
        display_post(posts_by_id[source_id], 'Original problem/source post')

    if show_all_replies:
        source_ids = set(item['relevant_post_ids'])
        candidates = [post for post in topic_posts if str(post['post_id']) not in source_ids]
    else:
        candidates = rank_answer_candidates(item, topic_posts, top_k=8, early_replies=3)
    if not candidates:
        display(Markdown('**No replies exist in this frozen thread. Label this query `None`.**'))
    for rank, candidate in enumerate(candidates, start=1):
        reason = candidate.get('candidate_reason', 'all_replies')
        score = candidate.get('review_score')
        extra = f'| candidate={reason}' + (f' | BM25={score:.2f}' if score is not None else '')
        display_post(candidate, f'Candidate {rank}', extra)


In [ ]:
QUERY_ID = 'kessy_start_button'  # Copy an id from the queue table.
SHOW_ALL_REPLIES = False
show_review(QUERY_ID, SHOW_ALL_REPLIES)


## 3. Record your human answer-post labels

Copy the `post_id` from every genuinely useful answer. Use a list for one or more relevant posts, `None` when no verified answer exists, and keep `[]` only while pending. Rerun this cell after editing so the values exist in the notebook kernel.


In [ ]:
HUMAN_ANSWER_LABELS = {
    'kessy_start_button': ['5308556', '5313848', '5724475'],
    'dpf_pressure_sensor': ['5920398'],
    'frequent_dpf_regen': ['5873012', '5875927', '5905591'],
    'adblue_limp_mode': ['5983756'],
    'electric_tailgate_partial_open': None,
    'replacement_battery_coding': ['5902151', '5902160'],
    'battery_parasitic_drain': ['5624618', '5624624', '5625616', '5625621'],
    'acc_vertical_alignment': None,
    'carplay_usb_no_power': None,
    'coolant_type_emergency': ['5936359', '5936368'],
    'unexplained_coolant_loss': ['5917595', '5917659'],
    'auxiliary_water_pump_location': None,
    'new_steering_wheel_retrofit': ['5793149', '5793394'],
    'door_contact_auto_hold_epb': ['5308956', '5625741'],
    'wheel_bearing_noise_constant': ['5754670', '5754707', '5754728', '5754927'],
    'aircon_refrigerant_2016': ['5783635'],
    'creaking_front_suspension': ['5904953'],
    'sunroof_creak_fix': ['5034440', '5888059', '5891211'],
    'tdi_oil_consumption': ['5862314', '5863239'],
    'cold_injector_deviation': None,
    'egr_p0401_after_regen': None,
    'key_fob_low_battery_warning': ['5995601', '5995750'],
    'reverse_camera_four_images': None,
}


## 4. Check review progress and label integrity

This provides fast feedback while you work. The strict exporter in the next section additionally checks that every selected post exists and belongs to the correct thread.


In [ ]:
progress_rows = []
for item in queries:
    value = HUMAN_ANSWER_LABELS.get(item['id'], [])
    status = 'no_verified_answer' if value is None else ('labelled' if value else 'pending')
    progress_rows.append({'id': item['id'], 'status': status, 'answer_post_ids': value})
progress = pd.DataFrame(progress_rows)
display(progress)
print(progress['status'].value_counts().to_dict())


## 5. Export only after all judgements are complete

Set the flag to `True` only after the progress table has no pending rows. The result is written outside Git first so it can be inspected before becoming the canonical evaluation file.


In [ ]:
EXPORT_REVIEWED_LABELS = True
if EXPORT_REVIEWED_LABELS:
    reviewed_dataset = build_reviewed_answer_eval(starter, HUMAN_ANSWER_LABELS, posts_by_id)
    REVIEWED_EVAL_PATH.write_text(json.dumps(reviewed_dataset, indent=2, ensure_ascii=False), encoding='utf-8')
    print(f'Exported {len(reviewed_dataset["queries"])} answer-labelled queries to:')
    print(REVIEWED_EVAL_PATH)
    print(f'Excluded as unanswerable: {len(reviewed_dataset["excluded_queries"])}')
else:
    print('Export disabled while review is in progress.')
